In [2]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
from transformers import pipeline

print("GPU disponível:", torch.cuda.is_available())

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU disponível: False


In [4]:
# A verificação retornou False para CUDA; o teste utilizará a CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float32,
    device=device,
)

print("Modelo carregado em:", device)

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 149.56it/s]


Modelo carregado em: cpu


In [5]:
#teste 1
import time
from transformers import GenerationConfig

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """Escreva uma frase comparando os tempos médios
até a sentença. Use somente os dados fornecidos.

Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Em 2025, o tempo médio para julgamento de acórdãos do Tribunal Superior Eleitoral (TSE) foi de 111,7 dias, enquanto o do Juizado Especial da Justiça Federal (JEF) era de 90,1 dias. A diferença entre esses dois períodos é de 21,6 dias.
Tempo: 13.8 segundos


In [ ]:
#teste 2 com refinamento do prompt
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """CONTEXTO
Os dados comparam o tempo médio até a sentença nos Juizados Especiais.
As siglas identificam os tribunais:
TJDFT: Distrito Federal.
TJSE: Sergipe.

TAREFA
Compare os tempos e informe qual tribunal apresentou o menor
tempo médio até a sentença e qual foi a diferença.
Mantenha exatamente as siglas TJDFT e TJSE, sem expandi-las.
Utilize somente as informações fornecidas.

DADOS
Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias

FORMATO
Uma única frase em português."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

A tarefa de comparar os tempos médios de sentença entre os Juizados Especiais no Distrito Federal (TJDFT) e no Estado do Sergipe (TJSE) é uma tarefa relevante para compreender a distribuição das penas aplicadas em diferentes regiões da justiça federal brasileira. A diferença de 21,6 dias entre os dois tribunais pode ser vista como um indicativo da complexidade e variação
Tempo: 13.6 segundos


In [7]:
#teste 3 com o mesmo refinamento do prompt anterior, testando um novo modelo

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

print("Modelo 1.5B carregado na CPU.")

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """CONTEXTO
Os dados comparam o tempo médio até a sentença nos Juizados Especiais.
As siglas identificam os tribunais:
TJDFT: Distrito Federal.
TJSE: Sergipe.

TAREFA
Compare os tempos e informe qual tribunal apresentou o menor
tempo médio até a sentença e qual foi a diferença.
Mantenha exatamente as siglas TJDFT e TJSE, sem expandi-las.
Utilize somente as informações fornecidas.

DADOS
Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias

FORMATO
Uma única frase em português."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carol\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hu

Modelo 1.5B carregado na CPU.
Em 2025, o Tribunal de Justiça do Estado de Sergipe (TJSE) apresentou o menor tempo médio até a sentença, sendo 111,7 dias contra 90,1 dias no Tribunal Regional do Direito do Distrito Federal e de Territórios (TJDFT). A diferença entre esses tempos é de 21,6 dias.
Tempo: 15.8 segundos


In [8]:
#teste 4 com novo prompt
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Em 2025, o Tribunal de Justiça Distrital Federal (TJDFT) tem um tempo médio até a sentença de 90,1 dias, enquanto o Tribunal de Justiça do Estado de São Paulo (TJSE) leva 111,7 dias, resultando em uma diferença de 21,6 dias.
Tempo: 18.3 segundos


In [9]:
#teste 5 com exemplo de saída
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Em 2025, o Tribunal de Justiça Distrital Federal (TJDFT) apresentou um tempo médio até a sentença de 90,1 dias, ficando 21,6 dias acima do Tribunal de Justiça Estadual (TJSE), que registrou 111,7 dias.
Tempo: 19.2 segundos


In [ ]:
#teste 6 com um novo modelo repetindo o teste anterior com exemplo de saída
import time
import torch
from transformers import pipeline, GenerationConfig

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

print("TinyLlama carregado na CPU.")

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carol\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 109.14it/s]
[transforme

TinyLlama carregado na CPU.
Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo
Tempo: 12.1 segundos


In [12]:
#Reexecução do teste 6 aumentando para 256 o limite de tokens
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo do TJSE, que registrou 111,7 dias.
Tempo: 17.3 segundos


In [ ]:
import time
import torch
from transformers import pipeline, GenerationConfig

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

#Reexecução do teste 6 
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:02<00:00, 92.11it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo do TJSE, que registrou 111,7 dias.
Tempo: 16.4 segundos


In [2]:
#Reexecução do teste 6 com outro tribunal
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJBA
Tempo do TJBA: 120,0 dias
Tribunal com maior tempo: TJMA
Tempo do TJMA: 125,7 dias
Diferença: 5,7 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJBA, 120,0 dias; maior tempo: TJMA, 125,7 dias; diferença: 5,7 dias.
Resposta: Em 2025, o TJBA apresentou tempo médio até a sentença de 120 dias, ficando 5,7 dias abaixo do TJMA, que registrou 125,7 dias.
Tempo: 21.2 segundos


In [3]:
print(
    "Memória dos parâmetros:",
    round(
        sum(p.numel() * p.element_size() for p in pipe.model.parameters())
        / 1000000000,
        2,
    ),
    "GB",
)

Memória dos parâmetros: 2.2 GB


In [4]:
# Teste de geração com a conexão de internet desativada.
inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJBA, 120,0 dias; maior tempo: TJMA, 125,7 dias; diferença: 5,7 dias.
Resposta: Em 2025, o TJBA apresentou tempo médio até a sentença de 120 dias, ficando 5,7 dias abaixo do TJMA, que registrou 125,7 dias.
Tempo: 16.5 segundos
